# 怎样把训练好的模型做成可调用的预测接口？：参考答案

每道题先写预期结果再运行。练习中的 TODO 需要自行完成；能顺序运行不代表已完成练习。

In [1]:
from pathlib import Path
CASE_DIR = Path.cwd()
# 载入本例的可修改函数和数据，所有算法都在同目录 solution.py。
exec(compile((CASE_DIR / "solution.py").read_text(encoding="utf-8"), str(CASE_DIR / "solution.py"), "exec"))

{
  "test_mae": 74.54137711788938,
  "valid_request_status": 200,
  "invalid_request_status": 422,
  "serialization_match": true,
  "sample_prediction": {
    "predicted_rent": 2399.72,
    "unit": "教学货币/月",
    "model_version": "teaching-v1"
  },
  "elapsed_seconds": 1.224
}


## 1. 补全：响应契约
调用接口，核对响应含预测值、单位和版本。

In [2]:
with TestClient(app) as c:
    r = c.post('/predict', json={'area': 60, 'rooms': 2, 'floor': 4})
assert r.status_code == 200 and set(r.json()) == {'predicted_rent', 'unit', 'model_version'}
print(r.json())

{'predicted_rent': 1883.54, 'unit': '教学货币/月', 'model_version': 'teaching-v1'}


## 2. 修改：缺失与类型错误
缺失floor、rooms传abc、面积0，均应返回422。

In [3]:
bad = [{'area': 60, 'rooms': 2}, {'area': 60, 'rooms': 'abc', 'floor': 2}, {'area': 0, 'rooms': 2, 'floor': 2}]
with TestClient(app) as c:
    for body in bad:
        assert c.post('/predict', json=body).status_code == 422
print('非法请求已拒绝')

非法请求已拒绝


## 3. 迁移：批量客户端
客户端顺序发送三条记录，一条输入对应一条输出。

In [4]:
requests = [{'area': a, 'rooms': 2, 'floor': 3} for a in [40, 60, 80]]
with TestClient(app) as c:
    responses = [c.post('/predict', json=body) for body in requests]
assert len(responses) == 3 and all((r.status_code == 200 for r in responses))
print([r.json()['predicted_rent'] for r in responses])

[1542.68, 1883.54, 2166.98]
